In [0]:
from pyspark.sql.functions import *

In [0]:
crm=spark.read.format("json")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .option("multiLine", "true")\
    .option("mode", "PERMISSIVE")\
    .load("s3://retail-lakehouse-ashu/raw/crm/crm_customers.json")

In [0]:
if "_corrupt_record" in crm.columns:
    crm.cache()
    crm.count()

    corrupt_count=crm.filter(col("_corrupt_record").isNotNull()).count()
    print(f"Corrupt Records: {corrupt_count}")
else:
    print("No _corrupt_record column present. JSON parsed successfully.")

In [0]:
crm.printSchema()

In [0]:
validation_condition=(
        col("customer_id").isNotNull() &
        col("email").isNotNull() &
        col("address.city").isNotNull() &
        col("address.pincode").isNotNull() &
        col("address.state").isNotNull() &
        col("name.first_name").isNotNull() &
        col("name.last_name").isNotNull() &
        col("phone").isNotNull() &
        col("registration_date").isNotNull()
        )

In [0]:
good_crm_df=crm.filter(validation_condition)

In [0]:
bad_crm_df=crm.filter(~validation_condition)

In [0]:
bronze_crm=good_crm_df.withColumn("ingestion_timestamp", current_timestamp())\
    .withColumn("source_system", lit("CRM"))\
    .withColumn("source_file", col("_metadata.file_path"))\
    .withColumn("batch_id", lit(1))

bronze_crm.display()

In [0]:
total_records=crm.count()

good_records=good_crm_df.count()

bad_records=bad_crm_df.count()

print(f"Total Records   : {total_records}")
print(f"Good Records    : {good_records}")
print(f"Bad Records     : {bad_records}")

In [0]:
duplicate_df=crm.groupBy(col("customer_id")).count().where(col("count")>1)
display(duplicate_df)

In [0]:
assert total_records==good_records+bad_records

In [0]:
bronze_crm.write \
    .format("delta") \
    .mode("overwrite") \
    .save("s3://retail-lakehouse-ashu/bronze/crm/")

In [0]:
bad_df=(
    bad_crm_df
        .withColumn("batch_id", lit(1))
        .withColumn("pipeline_name", lit("bronze_crm"))
        .withColumn("source_system", lit("CRM"))
        .withColumn("table_name", lit("CRM"))
        .withColumn("failure_reason", lit("MANDATORY_FIELD_VALIDATION_FAILED"))
        .withColumn("source_file", col("_metadata.file_path"))   # replace later with metadata
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("raw_record", to_json(struct(*bad_crm_df.columns)))
        .select(
        col("batch_id"),
        col("pipeline_name"),
        col("source_system"),
        col("table_name"),
        col("failure_reason"),
        col("source_file"),
        col("ingestion_timestamp"),
        col("raw_record")
    )
)

In [0]:
bad_df.write \
    .format("delta") \
    .mode("append") \
    .save("s3://retail-lakehouse-ashu/bad_records/")